# AgroMind v4: yield-grounded crop ranking + conformal prediction sets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ravixalgorithm/agromind/blob/main/notebooks/AgroMind_v4_Novel_Method.ipynb)

**Research question.** Published crop recommenders report ~99% accuracy, mostly on the Kaggle crop table split at random.
How good are they on **real Government of India data**, in **states and years the model has never seen**, and can we do better?

* **Data:** DES district × season × year crop statistics (1997-98 to 2022-23, official DES portal) + NASA POWER climate + SoilGrids soil (built by `backend/scripts/build_large_dataset.py`).
* **Task:** for each district-season-year, rank the crops grown there by realised yield percentile. The "best crop" is the top-yielding one.
* **Baselines (as in the literature):** Gaussian NB, Random Forest, XGBoost, MLP classifiers, plus naive *crop prior* and *last-year persistence*.
* **Ours (YieldRank):** LightGBM LambdaMART ranker on (district-season, crop) pairs, and **split-conformal prediction sets** with a 90% coverage target, calibrated on **held-out states**.
* **Protocols:** random 5-fold, **leave-states-out** (5 folds over states), **leave-years-out** (5 contiguous year blocks).

Everything below is computed live; nothing is pasted in. The code lives in `experiments/run.py` (fetched from GitHub on Colab).

In [1]:
# Setup: works locally (repo checkout) and on Colab (downloads from GitHub)
import os, sys, subprocess, urllib.request
from pathlib import Path
RAW = "https://raw.githubusercontent.com/ravixalgorithm/agromind/main/"
root = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "experiments/run.py").exists()), None)
if root is None:  # Colab
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm", "xgboost", "pyarrow"], check=True)
    root = Path("agromind");
    for f in ["experiments/run.py", "backend/data/large/agromind_india_large.parquet", "backend/data/Crop_recommendation.csv"]:
        (root / f).parent.mkdir(parents=True, exist_ok=True)
        if not (root / f).exists():
            urllib.request.urlretrieve(RAW + f, root / f)
sys.path.insert(0, str(root / "experiments"))
import numpy as np, pandas as pd, run
from scipy.stats import wilcoxon
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("repo root:", root.resolve())

repo root: /Users/originkit/.config/baton/worktrees/agromind-334992/claude-5


## 1. The literature's setting: Kaggle crop table, random 80/20 split
This reproduces the ~99% figures (2,200 rows, 100 per crop, no location or year).

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
k = pd.read_csv(root / "backend/data/Crop_recommendation.csv")
Xtr, Xte, ytr, yte = train_test_split(k.drop(columns="label"), k.label, test_size=0.2, stratify=k.label, random_state=0)
for name, m in [("Random Forest", RandomForestClassifier(300, random_state=0)), ("Gaussian NB", GaussianNB())]:
    print(f"{name:14s} accuracy on Kaggle random split: {100*(m.fit(Xtr, ytr).predict(Xte) == yte).mean():.2f}%")

Random Forest  accuracy on Kaggle random split: 99.55%
Gaussian NB    accuracy on Kaggle random split: 99.55%


## 2. Real data: DES district-season-year crop statistics

In [3]:
df = run.load()
crops = sorted(df.crop_slug.unique())
QX, Qy, Q = run.query_table(df, crops, hist=True)
print(f"(query, crop) rows : {len(df):,}")
print(f"queries            : {df.qid.nunique():,}  (district x season x year)")
print(f"crops / states     : {len(crops)} / {df.state.nunique()}   years {df.year.min()}-{df.year.max()}")
print("candidates per query:", df.groupby('qid').size().describe()[['mean','50%','max']].round(1).to_dict())
print("soil source share  :", df.soil_source.value_counts(normalize=True, dropna=False).round(3).to_dict())
df[['state','district','year','season','crop_slug','yield','yield_pct','temperature','rainfall','lag1_pct']].head(8)

(query, crop) rows : 259,181
queries            : 43,443  (district x season x year)
crops / states     : 49 / 36   years 1997-2022
candidates per query: {'mean': 6.0, '50%': 5.0, 'max': 27.0}
soil source share  : {'soilgrids_centroid': 0.915, 'state_median': 0.085, nan: 0.0}


,state,district,year,season,crop_slug,yield,yield_pct,temperature,rainfall,lag1_pct
0,Andaman and Nicobar Islands,Nicobars,2000,Kharif,arecanut,1.5949,0.7639,27.54,907.4,NaN
1,Andaman and Nicobar Islands,Nicobars,2000,Kharif,rice,3.1471,0.9107,27.54,907.4,NaN
2,Andaman and Nicobar Islands,Nicobars,2010,Rabi,arecanut,1.7055,0.6901,27.23,1256.3,NaN
3,Andaman and Nicobar Islands,Nicobars,2010,Rabi,cashewnut,0.2604,0.2619,27.23,1256.3,NaN
4,Andaman and Nicobar Islands,Nicobars,2011,Rabi,arecanut,1.1390,0.5063,27.67,886.9,0.6901
5,Andaman and Nicobar Islands,Nicobars,2011,Rabi,cashewnut,0.2750,0.4754,27.67,886.9,0.2619
6,Andaman and Nicobar Islands,Nicobars,2012,Rabi,arecanut,1.4200,0.6479,27.92,926.8,0.5063
7,Andaman and Nicobar Islands,Nicobars,2012,Rabi,cashewnut,0.2901,0.3500,27.92,926.8,0.4754


## 3. The method (source shown for transparency)
`fit_rank` = YieldRank (LambdaMART on yield-percentile relevance); `conformal` = LAC / APS split-conformal sets.

In [4]:
import inspect
print(inspect.getsource(run.fit_rank)); print(inspect.getsource(run.conformal))

def fit_rank(df, tr, crops, hist, objective):
    d = df.loc[tr]
    X = pair_features(d, crops, hist)
    rel = np.clip(np.floor(d.yield_pct * 10), 0, 9).astype(int)  # graded 0..9
    params = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)
    if objective == "lambdarank":
        m = lgb.LGBMRanker(objective="lambdarank", label_gain=list(range(10)), lambdarank_truncation_level=10, **params)
        m.fit(X, rel, group=d.groupby("qid", sort=False).size().values)
    else:
        m = lgb.LGBMRegressor(objective="regression", **params)
        m.fit(X, d.yield_pct)
    return lambda idx: m.predict(pair_features(df.loc[idx], crops, hist))

def conformal(cal, test):
    """cal/test: DataFrames with qid, p, best. LAC + APS sets; returns coverage, size per alpha."""
    res, per = {}, {}
    for method in ("LAC", "APS"):
        def scores(d):
          

In [5]:
MODELS = ["Persistence", "GaussianNB [6,12]", "RandomForest [8,21]", "XGBoost [11,22]", "MLP [13]", "YieldRank (ours)"]
rng = np.random.default_rng(run.SEED)
idx = lambda qs: df.index[df.qid.isin(qs)]

def evaluate(protocol):
    rows, perq, conf = [], [], []
    for f, (trq, teq, groups) in enumerate(run.folds(df, protocol)):
        a, b = next(run.GroupShuffleSplit(1, test_size=0.2, random_state=run.SEED + f).split(trq, groups=groups.loc[trq].values))
        grp_cal, rest = trq[b], trq[a]
        iid_cal = rng.choice(rest, size=len(grp_cal), replace=False)
        fit_i, te_i, gc_i, ic_i = idx(np.setdiff1d(rest, iid_cal)), idx(teq), idx(grp_cal), idx(iid_cal)
        for name in MODELS:
            score = run.MODELS[name](df, fit_i, crops, True, QX, Qy)
            s_te = score(te_i)
            pq = run.per_query(df.loc[te_i], s_te)
            s_gc, s_ic = score(gc_i), score(ic_i)
            T = run.fit_T(df.loc[ic_i, "qid"], s_ic, df.loc[ic_i, "best"].values)
            mk = lambda i, s: df.loc[i, ["qid", "best"]].assign(p=run.softmax_by_q(df.loc[i, "qid"], s, T))
            ci, per_i = run.conformal(mk(ic_i, s_ic), mk(te_i, s_te))
            cg, per_g = run.conformal(mk(gc_i, s_gc), mk(te_i, s_te))
            r = dict(fold=f, model=name.split(" [")[0], ndcg3=pq["ndcg@3"].mean(), hit1=100*pq["hit@1"].mean(), hit3=100*pq["hit@3"].mean(),
                     cov_iid=100*ci["LAC@0.9"]["coverage"], size_iid=ci["LAC@0.9"]["size"],
                     cov_heldout=100*cg["LAC@0.9"]["coverage"], size_heldout=cg["LAC@0.9"]["size"])
            rows.append(r)
            print(f"fold {f} | {r['model']:13s} | NDCG@3 {r['ndcg3']:.3f} | Hit@1 {r['hit1']:5.1f}% | Hit@3 {r['hit3']:5.1f}% | "
                  f"90% sets: iid-cal {r['cov_iid']:5.1f}% (size {r['size_iid']:.2f})  held-out-cal {r['cov_heldout']:5.1f}% (size {r['size_heldout']:.2f})", flush=True)
            perq.append(pq.assign(model=r["model"], fold=f).reset_index())
            conf.append(per_i.add_suffix("_iid").join(per_g.add_suffix("_grp")).assign(model=r["model"], fold=f,
                        state=lambda d: Q.state.reindex(d.index).values).reset_index(names="qid"))
    R = pd.DataFrame(rows)
    summ = R.groupby("model", sort=False)[["ndcg3", "hit1", "hit3", "cov_iid", "size_iid", "cov_heldout", "size_heldout"]].agg(["mean", "std"]).round(3)
    return R, summ, pd.concat(perq), pd.concat(conf)

def significance(P, base):
    o = P[P.model == "YieldRank (ours)"].set_index(["fold", "qid"])
    for b in base:
        x = P[P.model == b].set_index(["fold", "qid"]).reindex(o.index)
        for k in ["hit@1", "ndcg@3"]:
            d = (o[k] - x[k]).dropna()
            print(f"YieldRank vs {b:13s} {k:7s}: mean diff {d.mean():+.4f}   Wilcoxon p = {wilcoxon(d).pvalue:.2e}   (n = {len(d):,} test queries)")

## 4. Random split (optimistic, for reference)

In [6]:
R_rand, S_rand, P_rand, C_rand = evaluate("random")
S_rand

fold 0 | Persistence   | NDCG@3 0.915 | Hit@1  55.8% | Hit@3  87.4% | 90% sets: iid-cal  90.9% (size 3.18)  held-out-cal  90.3% (size 3.08)


fold 0 | GaussianNB    | NDCG@3 0.819 | Hit@1  32.4% | Hit@3  73.4% | 90% sets: iid-cal  90.3% (size 4.37)  held-out-cal  90.6% (size 4.40)


fold 0 | RandomForest  | NDCG@3 0.914 | Hit@1  57.8% | Hit@3  88.4% | 90% sets: iid-cal  90.3% (size 2.82)  held-out-cal  89.8% (size 2.76)


fold 0 | XGBoost       | NDCG@3 0.913 | Hit@1  58.0% | Hit@3  88.1% | 90% sets: iid-cal  90.6% (size 2.97)  held-out-cal  90.1% (size 2.89)


fold 0 | MLP           | NDCG@3 0.898 | Hit@1  53.9% | Hit@3  85.7% | 90% sets: iid-cal  90.3% (size 3.18)  held-out-cal  89.8% (size 3.11)


fold 0 | YieldRank (ours) | NDCG@3 0.936 | Hit@1  60.1% | Hit@3  90.2% | 90% sets: iid-cal  90.3% (size 2.66)  held-out-cal  89.9% (size 2.61)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.913 | Hit@1  58.1% | Hit@3  87.7% | 90% sets: iid-cal  89.8% (size 2.95)  held-out-cal  89.6% (size 2.91)


fold 4 | MLP           | NDCG@3 0.898 | Hit@1  54.1% | Hit@3  85.5% | 90% sets: iid-cal  89.6% (size 3.10)  held-out-cal  90.7% (size 3.26)


fold 4 | YieldRank (ours) | NDCG@3 0.936 | Hit@1  59.7% | Hit@3  89.8% | 90% sets: iid-cal  89.7% (size 2.61)  held-out-cal  90.3% (size 2.68)


ndcg3           hit1           hit3        cov_iid        size_iid        cov_heldout        size_heldout       
                   mean    std    mean    std    mean    std    mean    std     mean    std        mean    std         mean    std
model                                                                                                                             
Persistence       0.914  0.001  55.286  0.386  87.098  0.310  90.019  0.633    3.106  0.064      89.840  0.516        3.075  0.057
GaussianNB        0.819  0.002  33.092  0.506  73.001  0.403  90.042  0.459    4.349  0.042      90.436  0.425        4.393  0.050
RandomForest      0.915  0.001  57.650  0.642  88.475  0.080  89.971  0.422    2.778  0.041      90.054  0.291        2.785  0.040
XGBoost           0.913  0.001  57.448  0.709  87.807  0.315  90.086  0.422    2.920  0.044      90.072  0.366        2.915  0.034
MLP               0.900  0.002  53.785  0.428  85.673  0.305  89.973  0.336    3.155  0.040      89.998  0.714        3.163  0.076
YieldRank (ours)  0.936  0.000  59.641  0.328  89.842  0.350  90.038  0.273    2.636  0.030      89.971  0.395        2.622  0.036

## 5. Leave-states-out: train on some states, test on states never seen

In [7]:
R_lso, S_lso, P_lso, C_lso = evaluate("leave-states-out")
S_lso

fold 0 | Persistence   | NDCG@3 0.902 | Hit@1  51.0% | Hit@3  83.4% | 90% sets: iid-cal  88.9% (size 3.42)  held-out-cal  86.8% (size 3.14)


fold 0 | GaussianNB    | NDCG@3 0.741 | Hit@1  24.5% | Hit@3  56.4% | 90% sets: iid-cal  84.2% (size 4.85)  held-out-cal  86.5% (size 5.06)


fold 0 | RandomForest  | NDCG@3 0.814 | Hit@1  33.1% | Hit@3  67.3% | 90% sets: iid-cal  73.3% (size 3.68)  held-out-cal  85.5% (size 5.26)


fold 0 | XGBoost       | NDCG@3 0.826 | Hit@1  34.1% | Hit@3  68.8% | 90% sets: iid-cal  78.2% (size 3.81)  held-out-cal  89.2% (size 5.24)


fold 0 | MLP           | NDCG@3 0.817 | Hit@1  36.4% | Hit@3  69.9% | 90% sets: iid-cal  73.8% (size 3.45)  held-out-cal  88.6% (size 5.18)


fold 0 | YieldRank (ours) | NDCG@3 0.914 | Hit@1  52.0% | Hit@3  85.7% | 90% sets: iid-cal  90.2% (size 3.24)  held-out-cal  88.7% (size 3.02)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.839 | Hit@1  39.4% | Hit@3  73.5% | 90% sets: iid-cal  77.4% (size 3.36)  held-out-cal  89.4% (size 4.72)


fold 4 | MLP           | NDCG@3 0.828 | Hit@1  39.2% | Hit@3  74.0% | 90% sets: iid-cal  74.7% (size 3.08)  held-out-cal  85.1% (size 4.12)


fold 4 | YieldRank (ours) | NDCG@3 0.933 | Hit@1  59.9% | Hit@3  89.6% | 90% sets: iid-cal  90.4% (size 2.71)  held-out-cal  90.1% (size 2.67)


ndcg3           hit1           hit3        cov_iid        size_iid        cov_heldout        size_heldout       
                   mean    std    mean    std    mean    std    mean    std     mean    std        mean    std         mean    std
model                                                                                                                             
Persistence       0.909  0.021  54.535  4.316  86.277  3.480  89.033  3.321    3.074  0.264      88.810  3.008        3.051  0.226
GaussianNB        0.776  0.051  26.551  3.675  65.498  8.868  88.348  9.574    4.443  0.572      89.925  7.015        4.599  0.552
RandomForest      0.832  0.040  36.127  4.198  72.400  6.141  75.362  6.625    3.304  0.266      89.518  2.525        4.991  0.936
XGBoost           0.845  0.035  38.668  4.066  74.523  5.956  79.604  6.018    3.416  0.313      89.480  1.769        4.706  0.778
MLP               0.838  0.045  39.221  5.334  74.657  6.584  77.350  9.137    3.162  0.340      87.694  3.631        4.291  0.714
YieldRank (ours)  0.922  0.020  55.748  4.595  87.998  3.334  90.009  2.747    2.870  0.298      89.571  3.405        2.831  0.394

In [8]:
significance(P_lso, ["Persistence", "GaussianNB", "RandomForest", "XGBoost", "MLP"])

YieldRank vs Persistence   hit@1  : mean diff +0.0121   Wilcoxon p = 3.33e-08   (n = 43,443 test queries)
YieldRank vs Persistence   ndcg@3 : mean diff +0.0128   Wilcoxon p = 3.02e-126   (n = 43,443 test queries)
YieldRank vs GaussianNB    hit@1  : mean diff +0.2919   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs GaussianNB    ndcg@3 : mean diff +0.1465   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs RandomForest  hit@1  : mean diff +0.1962   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs RandomForest  ndcg@3 : mean diff +0.0907   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs XGBoost       hit@1  : mean diff +0.1708   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs XGBoost       ndcg@3 : mean diff +0.0773   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs MLP           hit@1  : mean diff +0.1653   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs MLP           ndcg@3 : mean diff

### Conformal coverage per held-out state (YieldRank, 90% target, LAC)
`cov_iid` = calibrated on random queries from training states; `cov_heldout` = calibrated on held-out training states.

In [9]:
c = C_lso[C_lso.model == "YieldRank (ours)"]
per_state = (c.groupby("state").agg(test_queries=("qid", "size"), cov_iid=("cov_LAC_iid", "mean"), cov_heldout=("cov_LAC_grp", "mean"),
                                     avg_set_size=("size_LAC_grp", "mean")))
per_state[["cov_iid", "cov_heldout"]] *= 100
per_state = per_state[per_state.test_queries >= 30].sort_values("test_queries", ascending=False).round(2)
print(f"states below 85% coverage: iid-cal {(per_state.cov_iid < 85).sum()} / {len(per_state)},  held-out-cal {(per_state.cov_heldout < 85).sum()} / {len(per_state)}")
per_state

states below 85% coverage: iid-cal 4 / 32,  held-out-cal 4 / 32


,test_queries,cov_iid,cov_heldout,avg_set_size
state,,,,
Uttar Pradesh,5349,91.38,89.81,3.12
Bihar,3596,91.35,89.82,2.39
Madhya Pradesh,3562,90.20,92.76,3.77
Karnataka,2729,86.22,85.34,2.71
Gujarat,2507,88.07,87.67,2.81
Rajasthan,2405,87.98,87.78,2.91
Maharashtra,2354,84.62,83.47,2.84
Odisha,2250,92.58,94.18,3.35
West Bengal,2060,91.84,90.63,2.51


## 6. Leave-years-out: train on some years, test on unseen year blocks

In [10]:
R_lyo, S_lyo, P_lyo, C_lyo = evaluate("leave-years-out")
S_lyo

fold 0 | Persistence   | NDCG@3 0.926 | Hit@1  59.9% | Hit@3  89.2% | 90% sets: iid-cal  92.2% (size 3.12)  held-out-cal  91.5% (size 3.01)


fold 0 | GaussianNB    | NDCG@3 0.813 | Hit@1  30.9% | Hit@3  72.1% | 90% sets: iid-cal  91.1% (size 4.46)  held-out-cal  91.4% (size 4.49)


fold 0 | RandomForest  | NDCG@3 0.904 | Hit@1  53.7% | Hit@3  86.5% | 90% sets: iid-cal  89.1% (size 3.00)  held-out-cal  89.9% (size 3.11)


fold 0 | XGBoost       | NDCG@3 0.906 | Hit@1  54.3% | Hit@3  86.6% | 90% sets: iid-cal  89.6% (size 3.12)  held-out-cal  90.4% (size 3.24)


fold 0 | MLP           | NDCG@3 0.898 | Hit@1  53.7% | Hit@3  85.5% | 90% sets: iid-cal  90.3% (size 3.28)  held-out-cal  90.3% (size 3.28)


fold 0 | YieldRank (ours) | NDCG@3 0.939 | Hit@1  60.6% | Hit@3  90.4% | 90% sets: iid-cal  91.5% (size 2.70)  held-out-cal  90.9% (size 2.62)


... (21 outputs left out)

fold 4 | XGBoost       | NDCG@3 0.896 | Hit@1  50.2% | Hit@3  83.7% | 90% sets: iid-cal  87.1% (size 3.14)  held-out-cal  91.9% (size 3.96)


fold 4 | MLP           | NDCG@3 0.888 | Hit@1  49.1% | Hit@3  82.9% | 90% sets: iid-cal  87.2% (size 3.20)  held-out-cal  91.6% (size 3.81)


fold 4 | YieldRank (ours) | NDCG@3 0.938 | Hit@1  58.5% | Hit@3  90.1% | 90% sets: iid-cal  89.6% (size 2.62)  held-out-cal  90.9% (size 2.82)


ndcg3           hit1           hit3        cov_iid        size_iid        cov_heldout        size_heldout       
                   mean    std    mean    std    mean    std    mean    std     mean    std        mean    std         mean    std
model                                                                                                                             
Persistence       0.914  0.014  55.180  3.544  87.019  1.912  90.063  1.788    3.118  0.065      89.687  1.670        3.061  0.221
GaussianNB        0.813  0.003  31.657  0.711  71.762  0.890  89.805  0.807    4.379  0.059      89.978  0.934        4.393  0.093
RandomForest      0.899  0.012  51.972  2.196  84.674  2.088  86.483  2.850    2.903  0.073      89.891  2.086        3.358  0.369
XGBoost           0.897  0.014  51.959  3.139  84.356  2.218  87.285  2.206    3.071  0.060      89.603  1.612        3.409  0.373
MLP               0.889  0.013  50.817  2.802  83.375  2.086  88.012  2.087    3.212  0.041      89.923  1.074        3.457  0.265
YieldRank (ours)  0.930  0.013  58.052  2.350  89.062  1.610  89.401  1.815    2.659  0.034      89.645  2.050        2.699  0.277

In [11]:
significance(P_lyo, ["Persistence", "GaussianNB", "RandomForest", "XGBoost", "MLP"])

YieldRank vs Persistence   hit@1  : mean diff +0.0285   Wilcoxon p = 3.23e-38   (n = 43,443 test queries)
YieldRank vs Persistence   ndcg@3 : mean diff +0.0163   Wilcoxon p = 7.61e-273   (n = 43,443 test queries)
YieldRank vs GaussianNB    hit@1  : mean diff +0.2638   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs GaussianNB    ndcg@3 : mean diff +0.1169   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs RandomForest  hit@1  : mean diff +0.0607   Wilcoxon p = 6.71e-150   (n = 43,443 test queries)
YieldRank vs RandomForest  ndcg@3 : mean diff +0.0313   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs XGBoost       hit@1  : mean diff +0.0609   Wilcoxon p = 1.24e-150   (n = 43,443 test queries)
YieldRank vs XGBoost       ndcg@3 : mean diff +0.0330   Wilcoxon p = 0.00e+00   (n = 43,443 test queries)
YieldRank vs MLP           hit@1  : mean diff +0.0721   Wilcoxon p = 1.34e-200   (n = 43,443 test queries)
YieldRank vs MLP           ndcg@3 : mean d

## 7. Summary: Hit@1 (%) and NDCG@3 by protocol

In [12]:
summary = pd.concat({"random": S_rand, "leave-states-out": S_lso, "leave-years-out": S_lyo}, axis=0)
summary[[("hit1", "mean"), ("hit1", "std"), ("ndcg3", "mean"), ("cov_iid", "mean"), ("size_iid", "mean"), ("cov_heldout", "mean"), ("size_heldout", "mean")]]

hit1         ndcg3 cov_iid size_iid cov_heldout size_heldout
                                     mean    std   mean    mean     mean        mean         mean
                 model                                                                           
random           Persistence       55.286  0.386  0.914  90.019    3.106      89.840        3.075
                 GaussianNB        33.092  0.506  0.819  90.042    4.349      90.436        4.393
                 RandomForest      57.650  0.642  0.915  89.971    2.778      90.054        2.785
                 XGBoost           57.448  0.709  0.913  90.086    2.920      90.072        2.915
                 MLP               53.785  0.428  0.900  89.973    3.155      89.998        3.163
                 YieldRank (ours)  59.641  0.328  0.936  90.038    2.636      89.971        2.622
leave-states-out Persistence       54.535  4.316  0.909  89.033    3.074      88.810        3.051
                 GaussianNB        26.551  3.675  0.776  88.348    4.443      89.925        4.599
                 RandomForest      36.127  4.198  0.832  75.362    3.304      89.518        4.991
                 XGBoost           38.668  4.066  0.845  79.604    3.416      89.480        4.706
                 MLP               39.221  5.334  0.838  77.350    3.162      87.694        4.291
                 YieldRank (ours)  55.748  4.595  0.922  90.009    2.870      89.571        2.831
leave-years-out  Persistence       55.180  3.544  0.914  90.063    3.118      89.687        3.061
                 GaussianNB        31.657  0.711  0.813  89.805    4.379      89.978        4.393
                 RandomForest      51.972  2.196  0.899  86.483    2.903      89.891        3.358
                 XGBoost           51.959  3.139  0.897  87.285    3.071      89.603        3.409
                 MLP               50.817  2.802  0.889  88.012    3.212      89.923        3.457
                 YieldRank (ours)  58.052  2.350  0.930  89.401    2.659      89.645        2.699

## 8. Check: the paper's numbers, recomputed here
Each value below is computed from this notebook's own runs and compared with `docs/research/results.json` → `v4.numbers`, which is the same file the paper's `numbers.tex` comes from.

In [13]:
import json
paper = json.load(open(root / "docs/research/results.json"))["v4"]["numbers"] if (root / "docs/research/results.json").exists() else {}
ours, ab = "YieldRank (ours)", {"random": "R", "leave-states-out": "LSO", "leave-years-out": "LYO"}
check = {}
for p, (S_, R_) in {"random": (S_rand, R_rand), "leave-states-out": (S_lso, R_lso), "leave-years-out": (S_lyo, R_lyo)}.items():
    m = R_.groupby("model").mean(numeric_only=True)
    base = m.drop(index=["YieldRank (ours)", "Persistence"]).hit1.idxmax()
    check |= {f"OursHit{ab[p]}": f"{m.loc[ours, 'hit1']:.1f}", f"OursNDCG{ab[p]}": f"{m.loc[ours, 'ndcg3']:.3f}",
              f"PersHit{ab[p]}": f"{m.loc['Persistence', 'hit1']:.1f}", f"BaseHit{ab[p]}": f"{m.loc[base, 'hit1']:.1f}", f"BaseName{ab[p]}": base.replace("RandomForest", "Random Forest").replace("GaussianNB", "Gaussian NB"),
              f"CovIID{ab[p]}": f"{m.loc[ours, 'cov_iid']:.1f}", f"SizeIID{ab[p]}": f"{m.loc[ours, 'size_iid']:.2f}",
              f"CovGRP{ab[p]}": f"{m.loc[ours, 'cov_heldout']:.1f}", f"SizeGRP{ab[p]}": f"{m.loc[ours, 'size_heldout']:.2f}"}
for k, v in check.items():
    print(f"{k:14s} notebook {v:>14s}   paper {paper.get(k, '?'):>14s}   {'OK' if paper.get(k) == v else 'DIFF'}")
print("all match:", all(paper.get(k) == v for k, v in check.items()))

OursHitR       notebook           59.6   paper           59.6   OK
OursNDCGR      notebook          0.936   paper          0.936   OK
PersHitR       notebook           55.3   paper           55.3   OK
BaseHitR       notebook           57.7   paper           57.7   OK
BaseNameR      notebook  Random Forest   paper  Random Forest   OK
CovIIDR        notebook           90.0   paper           90.0   OK
SizeIIDR       notebook           2.64   paper           2.64   OK
CovGRPR        notebook           90.0   paper           90.0   OK
SizeGRPR       notebook           2.62   paper           2.62   OK
OursHitLSO     notebook           55.7   paper           55.7   OK
OursNDCGLSO    notebook          0.922   paper          0.922   OK
PersHitLSO     notebook           54.5   paper           54.5   OK
BaseHitLSO     notebook           39.2   paper           39.2   OK
BaseNameLSO    notebook            MLP   paper            MLP   OK
CovIIDLSO      notebook           90.0   paper           90.0 

## Reading the numbers
* The ~99% of Section 1 is a property of the Kaggle table, not of crop choice. On real data, **no** model's top-1 pick matches the best-yielding crop anywhere near that often.
* Compare each baseline's Hit@1 between the random and leave-states-out tables: the drop is the over-optimism of random splits.
* *Persistence* (repeat last year's yield ranking) is a strong, rarely reported baseline. The gap between it and YieldRank is the honest size of the learned gain.
* Conformal coverage with i.i.d. calibration is a guarantee about *seen* states. Per-state coverage shows how it behaves on unseen ones.

The paper (`paper/main.pdf`) reports the same experiment from `experiments/run.py`.